# Lezione 2 — Costruisci la tua prima rete neurale da zero

**Obiettivi di oggi**
- Capire cos'è un tensore e perché TensorFlow ha questo nome.
- Definire manualmente i pesi e i bias di una rete densa.
- Addestrarla con `tf.GradientTape` e confrontare il risultato con la Keras della lezione precedente.

Consulta il [capitolo 1 del manuale](../manual/01_fundamentos.md) per la spiegazione completa.

## 1. Cos'è un tensore?

Un tensore è un array multidimensionale: un numero singolo è un tensore di dimensione zero, una lista è di dimensione uno, un'immagine in bianco e nero è di dimensione due. TensorFlow rappresenta così sia i tuoi dati che i pesi interni della rete.

In [ ]:
import tensorflow as tf

escalar = tf.constant(7)
vector = tf.constant([1, 2, 3])
matriz = tf.constant([[1, 2], [3, 4]])

print("Escalar (dimensión 0):", escalar.shape)
print("Vector (dimensión 1):", vector.shape)
print("Matriz (dimensión 2):", matriz.shape)

## 2. Carica gli stessi dati della lezione precedente, in formato "appiattito"

In [ ]:
(x_entreno, y_entreno), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_entreno = (x_entreno / 255.0).reshape(-1, 784).astype("float32")
x_test = (x_test / 255.0).reshape(-1, 784).astype("float32")

y_entreno_onehot = tf.keras.utils.to_categorical(y_entreno, 10)
y_test_onehot = tf.keras.utils.to_categorical(y_test, 10)

print("Forma de x_entreno:", x_entreno.shape)

## 3. Definisci pesi e bias come variabili addestrabili

Niente `Dense`: creerai tu stesso la matrice dei pesi e il vettore dei bias.

In [ ]:
pesos = tf.Variable(tf.random.normal([784, 10], stddev=0.1))
sesgos = tf.Variable(tf.zeros([10]))

def prediccion(x):
    return tf.nn.softmax(tf.matmul(x, pesos) + sesgos)

## 4. Addestra con `tf.GradientTape`

Il "nastro" registra le operazioni mentre vengono eseguite, in modo da poter calcolare successivamente il gradiente di ogni peso: la direzione in cui spostarlo per ridurre l'errore.

In [ ]:
optimizador = tf.keras.optimizers.Adam(learning_rate=0.01)
tamano_lote = 128

for epoca in range(3):
    for i in range(0, len(x_entreno), tamano_lote):
        x_lote = x_entreno[i:i + tamano_lote]
        y_lote = y_entreno_onehot[i:i + tamano_lote]
        with tf.GradientTape() as cinta:
            y_predicha = prediccion(x_lote)
            perdida = tf.keras.losses.categorical_crossentropy(y_lote, y_predicha)
        gradientes = cinta.gradient(perdida, [pesos, sesgos])
        optimizador.apply_gradients(zip(gradientes, [pesos, sesgos]))
    print(f"Época {epoca + 1} completada")

## 5. Valuta la tua rete manuale

In [ ]:
predicciones_test = prediccion(x_test)
aciertos = tf.equal(tf.argmax(predicciones_test, axis=1), tf.argmax(y_test_onehot, axis=1))
exactitud = tf.reduce_mean(tf.cast(aciertos, tf.float32))
print(f"Exactitud sobre el conjunto de test: {exactitud.numpy():.2%}")

## Riepilogo

- Una rete con un solo strato denso, costruita manualmente, riconosce già le cifre con abbastanza precisione (anche se meno della rete a due strati di Keras della lezione 1).
- `tf.Variable` salva i pesi addestrabili; `tf.GradientTape` calcola i loro gradienti; l'optimizer li aggiorna.
- Questo è esattamente quello che Keras fa per te quando chiami `fit`: ora sai cosa c'è dentro.

Continua con il [notebook di pratica di questa lezione](../notebooks_practica/clase02_practica_red_manual.ipynb).